# Cache Hierarchy & Branch Predictor Simulator — Final Kaggle Runner
CPU only; no GPU needed. This notebook regenerates project source, tests, synthetic traces, 21 experiment configurations, and plots.


In [ ]:
from pathlib import Path
files = {'src/main.cpp': '#include <algorithm>\n#include <cstdint>\n#include <cstdlib>\n#include <fstream>\n#include <iomanip>\n#include <iostream>\n#include <list>\n#include <memory>\n#include <sstream>\n#include <stdexcept>\n#include <string>\n#include <vector>\nusing u64=uint64_t;\nstruct Config {size_t size=32768, line=64, ways=8; std::string policy="lru";};\nstruct Stats {u64 accesses=0,hits=0,misses=0;};\nstruct Line {bool valid=false;u64 tag=0;u64 last=0;};\nclass Cache {\n Config c; size_t sets; std::vector<std::vector<Line>> data; std::vector<std::vector<bool>> tree; u64 clock=0;\n size_t victim(size_t s){\n  if(c.policy=="lru") { for(size_t i=0;i<c.ways;i++) if(!data[s][i].valid) return i; size_t v=0;for(size_t i=1;i<c.ways;i++)if(data[s][i].last<data[s][v].last)v=i;return v; }\n  for(size_t i=0;i<c.ways;i++) if(!data[s][i].valid)return i;\n  size_t node=0,way=0;for(size_t span=c.ways/2;span;span/=2){bool go=tree[s][node];way+=go?span:0;node=2*node+1+(go?1:0);}return way;\n }\n void touch(size_t s,size_t w){data[s][w].last=++clock; if(c.policy=="plru"){size_t node=0;for(size_t span=c.ways/2;span;span/=2){bool right=(w&span)!=0;tree[s][node]=!right;node=2*node+1+(right?1:0);}}}\n public:Stats stats;\n explicit Cache(Config x):c(x){if(!c.size||!c.line||!c.ways||c.ways>c.size/c.line||c.size%(c.line*c.ways))throw std::runtime_error("Invalid cache geometry");sets=c.size/(c.line*c.ways);if(c.policy!="lru"&&c.policy!="plru")throw std::runtime_error("Policy must be lru or plru");if(c.policy=="plru"&&(c.ways&(c.ways-1)))throw std::runtime_error("PLRU requires power-of-two ways");data.resize(sets,std::vector<Line>(c.ways));if(c.policy=="plru")tree.resize(sets,std::vector<bool>(c.ways-1,false));}\n bool probe(u64 addr){stats.accesses++;u64 block=addr/c.line;size_t set=block%sets;u64 tag=block/sets;for(size_t w=0;w<c.ways;w++)if(data[set][w].valid&&data[set][w].tag==tag){stats.hits++;touch(set,w);return true;}stats.misses++;return false;}\n void insert(u64 addr){u64 block=addr/c.line;size_t set=block%sets;u64 tag=block/sets;for(size_t w=0;w<c.ways;w++)if(data[set][w].valid&&data[set][w].tag==tag){touch(set,w);return;}size_t w=victim(set);data[set][w]={true,tag,0};touch(set,w);}\n};\nclass Predictor {public:virtual ~Predictor()=default;virtual bool predict(u64 pc)=0;virtual void update(u64 pc,bool taken)=0;};\nclass Bimodal:public Predictor {std::vector<uint8_t> t;public:explicit Bimodal(size_t bits):t(1ULL<<bits,1){}bool predict(u64 pc)override{return t[(pc>>2)&(t.size()-1)]>=2;}void update(u64 pc,bool y)override{auto &x=t[(pc>>2)&(t.size()-1)];x=y?std::min<int>(3,x+1):std::max<int>(0,x-1);}};\nclass Gshare:public Predictor {std::vector<uint8_t> t;u64 hist=0,mask;public:explicit Gshare(size_t bits):t(1ULL<<bits,1),mask((1ULL<<bits)-1){}bool predict(u64 pc)override{return t[((pc>>2)^hist)&mask]>=2;}void update(u64 pc,bool y)override{auto &x=t[((pc>>2)^hist)&mask];x=y?std::min<int>(3,x+1):std::max<int>(0,x-1);hist=((hist<<1)|y)&mask;}};\nclass Tournament:public Predictor {Bimodal b;Gshare g;std::vector<uint8_t> choice;public:explicit Tournament(size_t bits):b(bits),g(bits),choice(1ULL<<bits,1){}bool predict(u64 pc)override{size_t i=(pc>>2)&(choice.size()-1);return choice[i]>=2?g.predict(pc):b.predict(pc);}void update(u64 pc,bool y)override{size_t i=(pc>>2)&(choice.size()-1);bool bp=b.predict(pc),gp=g.predict(pc);if(bp!=gp){auto &v=choice[i];v=gp==y?std::min<int>(3,v+1):std::max<int>(0,v-1);}b.update(pc,y);g.update(pc,y);}};\nstatic u64 number(const std::string&s){size_t i=0;auto v=std::stoull(s,&i,0);if(i!=s.size())throw std::runtime_error("Bad integer "+s);return v;}\nstatic void help(){std::cout<<"Usage: simulator --trace path [--l1-size bytes] [--l2-size bytes] [--l3-size bytes] [--line bytes] [--ways n] [--policy lru|plru] [--predictor bimodal|gshare|tournament] [--history bits] [--out report.csv]\\nTrace lines: M <address>, B <pc> <T|N>, I <count>; addresses decimal or 0x hex; # comments. I records supply dynamic instruction counts for MPKI.\\n";}\nint main(int argc,char**argv){try{\n Config c;std::string file,out="results.csv",prediction="tournament";size_t history=10;size_t sizes[3]={32768,262144,8388608};\n for(int i=1;i<argc;i++){std::string k=argv[i];if(k=="--help"){help();return 0;}if(i+1>=argc)throw std::runtime_error("Missing argument for "+k);std::string v=argv[++i];if(k=="--trace")file=v;else if(k=="--out")out=v;else if(k=="--l1-size")sizes[0]=number(v);else if(k=="--l2-size")sizes[1]=number(v);else if(k=="--l3-size")sizes[2]=number(v);else if(k=="--line")c.line=number(v);else if(k=="--ways")c.ways=number(v);else if(k=="--policy")c.policy=v;else if(k=="--predictor")prediction=v;else if(k=="--history")history=number(v);else throw std::runtime_error("Unknown option: "+k);}\n if(file.empty())throw std::runtime_error("--trace required");\n if(history<1||history>20)throw std::runtime_error("history must be 1..20");\n std::vector<std::unique_ptr<Cache>> caches;for(auto sz:sizes){c.size=sz;caches.emplace_back(std::make_unique<Cache>(c));}\n std::unique_ptr<Predictor> pred;if(prediction=="bimodal")pred=std::make_unique<Bimodal>(history);else if(prediction=="gshare")pred=std::make_unique<Gshare>(history);else if(prediction=="tournament")pred=std::make_unique<Tournament>(history);else throw std::runtime_error("Unknown predictor");\n std::ifstream in(file);if(!in)throw std::runtime_error("Cannot open "+file);u64 mem=0,branches=0,correct=0,instructions=0;bool explicitInstructions=false;std::string line;size_t lineNum=0;\n while(std::getline(in,line)){lineNum++;auto comment=line.find(\'#\');if(comment!=std::string::npos)line.resize(comment);std::istringstream ss(line);std::string type,a,b,extra;if(!(ss>>type))continue;\n  if(type=="M"){if(!(ss>>a)||(ss>>extra))throw std::runtime_error("Invalid M record");u64 address=number(a);mem++;size_t level=0;for(;level<caches.size();level++)if(caches[level]->probe(address))break; // refill previously missed levels; no enforced inclusion or coherence\n   for(size_t j=0;j<level;j++)caches[j]->insert(address);\n  }else if(type=="B"){if(!(ss>>a>>b)||(ss>>extra)||(b!="T"&&b!="N"))throw std::runtime_error("Invalid B record");u64 pc=number(a);bool y=b=="T";branches++;correct+=pred->predict(pc)==y;pred->update(pc,y);\n  }else if(type=="I"){if(!(ss>>a)||(ss>>extra))throw std::runtime_error("Invalid I record");explicitInstructions=true;instructions+=number(a);\n  }else throw std::runtime_error("Invalid record type");\n }\n if(!explicitInstructions)instructions=mem+branches; // explicit proxy, NOT true instruction count\n if(in.bad())throw std::runtime_error("Error reading trace");\n std::ofstream os(out);if(!os)throw std::runtime_error("Cannot write "+out);os<<"metric,value\\n";auto write=[&](std::string name,double val){os<<name<<","<<std::setprecision(12)<<val<<"\\n";};write("memory_accesses",mem);write("branches",branches);write("instructions",instructions);write("instruction_count_estimated",!explicitInstructions);for(size_t j=0;j<3;j++){auto &s=caches[j]->stats;std::string p="l"+std::to_string(j+1)+"_";write(p+"accesses",s.accesses);write(p+"hits",s.hits);write(p+"misses",s.misses);write(p+"local_miss_rate",s.accesses?double(s.misses)/s.accesses:0);write(p+"mpki",instructions?1000.0*s.misses/instructions:0);}write("branch_correct",correct);write("branch_accuracy",branches?double(correct)/branches:0);write("branch_mpki",instructions?1000.0*(branches-correct)/instructions:0);\n std::cout<<"Trace "<<file<<"; memory="<<mem<<", branches="<<branches<<", instructions="<<instructions<<(explicitInstructions?" (trace)":" (proxy)")<<"\\n";for(size_t j=0;j<3;j++){auto&s=caches[j]->stats;std::cout<<"L"<<j+1<<" accesses="<<s.accesses<<" misses="<<s.misses<<" local_miss_rate="<<std::fixed<<std::setprecision(4)<<(s.accesses?double(s.misses)/s.accesses:0)<<"\\n";}std::cout<<"Branch accuracy="<<(branches?double(correct)/branches:0)<<"; CSV: "<<out<<"\\n";\n }catch(const std::exception&e){std::cerr<<"Error: "<<e.what()<<"\\n";return 1;}}\n', 'scripts/generate_trace.py': '#!/usr/bin/env python3\n"""Deterministic synthetic trace. Not representative of SPEC workloads."""\nimport argparse, random\np=argparse.ArgumentParser();p.add_argument(\'--output\',default=\'traces/synthetic.trace\');p.add_argument(\'--accesses\',type=int,default=50000);p.add_argument(\'--seed\',type=int,default=42);args=p.parse_args()\nrng=random.Random(args.seed)\nwith open(args.output,\'w\') as f:\n    f.write(\'# synthetic sequential/hot/random accesses and correlated branches\\n\')\n    f.write(f\'I {args.accesses*4}\\n\')\n    for i in range(args.accesses):\n        mode=i%100\n        addr=(i%2048)*64 if mode<65 else (rng.randrange(128)*64 if mode<90 else rng.randrange(65536)*64)\n        f.write(f\'M 0x{addr:x}\\n\')\n        if i%4==0:\n            pc=0x1000+((i//4)%16)*4\n            taken=(i//4)%16 < 12 if rng.random()<0.9 else bool(rng.getrandbits(1))\n            f.write(f\'B 0x{pc:x} {"T" if taken else "N"}\\n\')\nprint(args.output)\n', 'scripts/sweep.py': "import csv, pathlib, subprocess, argparse\np=argparse.ArgumentParser();p.add_argument('--sim',default='./simulator');p.add_argument('--trace',default='traces/synthetic.trace');p.add_argument('--out',default='results/sweep.csv');args=p.parse_args()\npathlib.Path(args.out).parent.mkdir(parents=True,exist_ok=True)\nrows=[]\nfor ways in [1,2,4,8]:\n for policy in ['lru','plru']:\n  if policy=='plru' and ways==1:continue\n  for predictor in ['bimodal','gshare','tournament']:\n   temp=str(pathlib.Path(args.out).parent/'one_run.csv')\n   cmd=[args.sim,'--trace',args.trace,'--ways',str(ways),'--policy',policy,'--predictor',predictor,'--out',temp]\n   subprocess.run(cmd,check=True,stdout=subprocess.DEVNULL)\n   metrics={r['metric']:r['value'] for r in csv.DictReader(open(temp))}\n   rows.append(dict(ways=ways,policy=policy,predictor=predictor,**metrics))\nwith open(args.out,'w',newline='') as f:\n w=csv.DictWriter(f,fieldnames=rows[0].keys());w.writeheader();w.writerows(rows)\nprint('Saved',args.out,'with',len(rows),'configurations')\n", 'scripts/plot.py': "import csv,matplotlib.pyplot as plt\nrows=list(csv.DictReader(open('results/sweep.csv')))\nrows=[r for r in rows if r['policy']=='lru' and r['predictor']=='tournament']\nways=[int(r['ways']) for r in rows]\nfor field,label,file in [('l2_local_miss_rate','L2 local miss rate','results/cache_miss_rate.png'),('branch_accuracy','Branch accuracy','results/branch_accuracy.png')]:\n fig,ax=plt.subplots();ax.plot(ways,[float(r[field]) for r in rows],marker='o');ax.set_xticks(ways);ax.set_xlabel('Associativity (ways)');ax.set_ylabel(label);ax.set_title('Synthetic trace — illustrative, not SPEC');ax.grid(True,alpha=.2);fig.savefig(file,dpi=160,bbox_inches='tight');plt.close(fig);print(file)\n", 'tests/test_sim.py': "import csv, pathlib, subprocess, tempfile, unittest, random\nROOT=pathlib.Path(__file__).resolve().parents[1]\nclass Tests(unittest.TestCase):\n def run_trace(self, lines, *args):\n  with tempfile.TemporaryDirectory() as d:\n   t=pathlib.Path(d)/'t.trace';o=pathlib.Path(d)/'o.csv';t.write_text(lines)\n   p=subprocess.run([str(ROOT/'simulator'),'--trace',str(t),'--out',str(o),*args],capture_output=True,text=True)\n   self.assertEqual(p.returncode,0,p.stderr)\n   with o.open() as f:\n    return {r['metric']:float(r['value']) for r in csv.DictReader(f)}\n def run_failure(self, lines, *args):\n  with tempfile.TemporaryDirectory() as d:\n   t=pathlib.Path(d)/'t.trace';o=pathlib.Path(d)/'o.csv';t.write_text(lines)\n   return subprocess.run([str(ROOT/'simulator'),'--trace',str(t),'--out',str(o),*args],capture_output=True,text=True)\n def test_empty_trace(self):\n  m=self.run_trace('# empty\\n')\n  self.assertEqual(m['memory_accesses'],0)\n  self.assertEqual(m['branch_accuracy'],0)\n def test_invalid_trace_fails(self):\n  self.assertNotEqual(self.run_failure('M not_an_address\\n').returncode,0)\n  self.assertNotEqual(self.run_failure('B 0x100 X\\n').returncode,0)\n def test_invalid_geometry_fails(self):\n  self.assertNotEqual(self.run_failure('M 0\\n','--l1-size','64','--ways','8').returncode,0)\n  self.assertNotEqual(self.run_failure('M 0\\n','--ways','3','--policy','plru').returncode,0)\n def test_exact_bimodal_behavior(self):\n  m=self.run_trace('I 100\\n'+'B 0x100 T\\n'*4,'--predictor','bimodal','--history','2')\n  self.assertEqual(m['branch_correct'],3)\n  self.assertEqual(m['branch_accuracy'],.75)\n  self.assertEqual(m['branch_mpki'],10)\n def test_cache_lru_against_independent_oracle(self):\n  rng=random.Random(2030)\n  addresses=[rng.randrange(0,12)*64 for _ in range(400)]\n  # Single set, two ways, two cache lines; independent MRU list implementation.\n  lines=[];misses=0\n  for a in addresses:\n   if a in lines:lines.remove(a)\n   else:\n    misses+=1\n    if len(lines)==2:lines.pop()\n   lines.insert(0,a)\n  tr='I 1000\\n'+''.join(f'M {a}\\n' for a in addresses)\n  m=self.run_trace(tr,'--l1-size','128','--l2-size','128','--l3-size','128','--ways','2')\n  self.assertEqual(m['l1_misses'],misses)\n  self.assertEqual(m['l1_hits'],400-misses)\n  self.assertEqual(m['l1_mpki'],misses)\n def test_hierarchy_refills(self):\n  # L1 direct-mapped two lines; L2 four lines. A,B,A => third L1 miss and L2 hit.\n  tr='I 100\\nM 0\\nM 128\\nM 0\\n'\n  m=self.run_trace(tr,'--l1-size','128','--l2-size','256','--l3-size','512','--ways','1')\n  self.assertEqual(m['l1_misses'],3)\n  self.assertEqual(m['l2_misses'],2)\n  self.assertEqual(m['l2_hits'],1)\n def test_compulsory_and_hit(self):\n  m=self.run_trace('I 100\\nM 0x0\\nM 0x4\\nM 0x40\\nM 0x0\\n')\n  self.assertEqual(m['l1_misses'],2);self.assertEqual(m['l1_hits'],2);self.assertEqual(m['l2_misses'],2);self.assertEqual(m['l3_misses'],2);self.assertEqual(m['l1_mpki'],20)\n def test_lru_eviction(self):\n  m=self.run_trace('M 0x0\\nM 0x80\\nM 0x0\\nM 0x100\\nM 0x80\\n','--l1-size','128','--l2-size','128','--l3-size','128','--ways','2')\n  self.assertEqual(m['l1_misses'],4)\n def test_predictor(self):\n  m=self.run_trace('I 100\\n'+'B 0x100 T\\n'*12,'--predictor','bimodal')\n  self.assertEqual(m['branches'],12);self.assertGreater(m['branch_accuracy'],.8)\n def test_plru(self):\n  m=self.run_trace('M 0x0\\nM 0x40\\nM 0x80\\nM 0xc0\\nM 0x0\\n','--ways','4','--policy','plru')\n  self.assertEqual(m['l1_misses'],4)\nif __name__=='__main__':unittest.main()\n"}
for name, data in files.items():
 p=Path(name); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(data)
Path("results").mkdir(exist_ok=True)
Path("traces").mkdir(exist_ok=True)
print("Project sources created")


In [ ]:
!g++ -O2 -std=c++17 -Wall -Wextra src/main.cpp -o simulator


In [ ]:
!python3 -m unittest discover -s tests -v


In [ ]:
!python3 scripts/generate_trace.py --output traces/synthetic.trace --accesses 50000 --seed 42


In [ ]:
!./simulator --trace traces/synthetic.trace --out results/base.csv


In [ ]:
!python3 scripts/sweep.py --sim ./simulator --trace traces/synthetic.trace --out results/sweep.csv


In [ ]:
!python3 scripts/plot.py


In [ ]:
import pandas as pd
print(pd.read_csv("results/base.csv").to_string(index=False))
display(pd.read_csv("results/sweep.csv").head(10))


In [ ]:
from IPython.display import display, Image
display(Image(filename="results/cache_miss_rate.png"))
display(Image(filename="results/branch_accuracy.png"))


In [ ]:
import zipfile
with zipfile.ZipFile("cache_sim_results.zip","w",compression=zipfile.ZIP_DEFLATED) as z:
 for p in Path("results").glob("*"):
  if p.is_file():z.write(p)
print("Saved cache_sim_results.zip")
